In [ ]:
# ============================================================
# Notebook 06: "When the Geometry Lies"
# Regression from the Inside: Seeing the Geometry of Linear Models
# ============================================================

# --- Environment setup (run this cell first) ---
import sys

# Install regression_geometry package if not available
try:
    import regression_geometry
except ImportError:
    # Running in Colab or fresh environment — install from GitHub
    print("Installing regression_geometry package...")
    !pip install -q git+https://github.com/YOUR_USERNAME/regression-geometry-course.git
    print("Done! If you see import errors below, restart the runtime (Runtime → Restart) and run this cell again.")

# --- Standard imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import linalg

import statsmodels.api as sm

from regression_geometry.core import ColumnSpace, Projection, HatMatrix, Ellipsoid
from regression_geometry.core import frisch_waugh_lovell, angle_between, demean
from regression_geometry.data import load_meridian
from regression_geometry.colors import *

# --- Rendering backend toggle ---
INTERACTIVE = True
try:
    from regression_geometry import interactive as viz_mod
    if not viz_mod.AVAILABLE:
        INTERACTIVE = False
except ImportError:
    INTERACTIVE = False

if INTERACTIVE:
    from regression_geometry import interactive as viz
else:
    from regression_geometry import plots as viz

from regression_geometry.scoreboard import GeometricScoreboard

# --- Reproducibility ---
np.random.seed(42)

> *"A map that is perfectly drawn can still point you to the wrong city."*

## When the Geometry Lies

Every notebook so far has made regression clearer. The geometry revealed what coefficients are (Notebook 1), why residuals are perpendicular (Notebook 2), where R² comes from (Notebook 3), what "controlling for" means (Notebook 4), and who influences the projection (Notebook 5). You've built a geometric lens, and through it, regression has become transparent.

This notebook is about the moment that lens fails.

Two different failures. One you can see coming. One you can't.

## Part 1: Multicollinearity — The Geometry Breaks

In [ ]:
# Generate synthetic data with 3 predictors and moderate correlations
rng = np.random.default_rng(seed=123)
n = 200

mean = [0, 0, 0]
cov_low = [[1.0, 0.3, 0.3],
            [0.3, 1.0, 0.3],
            [0.3, 0.3, 1.0]]
X_raw = rng.multivariate_normal(mean, cov_low, size=n)

# True model: y = 2*x1 + 3*x2 + 1.5*x3 + noise
beta_true = np.array([2.0, 3.0, 1.5])
y_synth = X_raw @ beta_true + rng.normal(0, 2.0, size=n)

# Fit OLS and show geometric diagnostics
X_low = sm.add_constant(X_raw)
model_low = sm.OLS(y_synth, X_low).fit()
print(model_low.summary())

cs_low = ColumnSpace(X_raw, add_intercept=True)
proj_low = cs_low.project(y_synth)
print(f"\nCondition number: {cs_low.condition_number():.1f}")
print(f"R²: {proj_low.r_squared:.3f}")

scoreboard_low = GeometricScoreboard(
    proj=proj_low, cs=cs_low,
    active_gauges=["theta", "r_squared", "leverage", "residual_norm", "kappa"],
    mode="widget" if INTERACTIVE else "static"
)
scoreboard_low.display()

In [ ]:
# 3D projection for a 3-observation subset — column space is wide and stable
cs_sub = ColumnSpace(X_raw[:3], add_intercept=True)
fig = viz.plot_projection_3d(cs_sub, y_synth[:3], title="Well-conditioned column space")

This looks good. Every diagnostic says this regression is solid. The column space is well-conditioned, the projection sits firmly on the plane.

---
### 🛑 PREDICT FIRST

We're going to increase the correlation between x₁ and x₂ from 0.3 to 0.98, while keeping their correlations with x₃ and y the same.

**Before running the next cell, write your prediction below:**

(a) What will happen to the shape of the column space?

(b) What will happen to the individual coefficient estimates for x₁ and x₂?

(c) What will happen to the R²?

(d) What will happen to the standard errors of β̂₁ and β̂₂?

---

In [ ]:
# YOUR PREDICTION:
#
#

In [ ]:
# Regenerate data with r(x1, x2) = 0.98 — same true model
cov_high = [[1.0, 0.98, 0.3],
             [0.98, 1.0, 0.3],
             [0.3,  0.3, 1.0]]
X_high_raw = rng.multivariate_normal(mean, cov_high, size=n)
y_high = X_high_raw @ beta_true + rng.normal(0, 2.0, size=n)

X_high = sm.add_constant(X_high_raw)
model_high = sm.OLS(y_high, X_high).fit()
print(model_high.summary())

cs_high = ColumnSpace(X_high_raw, add_intercept=True)
proj_high = cs_high.project(y_high)

print(f"\n{'':30s} {'Low corr':>10s}  {'High corr':>10s}")
print(f"{'R²':30s} {proj_low.r_squared:10.3f}  {proj_high.r_squared:10.3f}")
print(f"{'SE(β₁)':30s} {model_low.bse[1]:10.3f}  {model_high.bse[1]:10.3f}")
print(f"{'SE(β₂)':30s} {model_low.bse[2]:10.3f}  {model_high.bse[2]:10.3f}")
print(f"{'Condition number':30s} {cs_low.condition_number():10.1f}  {cs_high.condition_number():10.1f}")

In [ ]:
# Side-by-side: well-conditioned vs collinear column spaces
cs_low_3 = ColumnSpace(X_raw[:3], add_intercept=True)
cs_high_3 = ColumnSpace(X_high_raw[:3], add_intercept=True)
fig = viz.plot_collinearity_comparison(
    cs_low_3, cs_high_3, y_synth[:3],
    titles=("r = 0.3 (stable)", "r = 0.98 (degenerate)")
)

R² didn't change. The model explains the same amount. But the individual coefficients for x₁ and x₂ are now unreliable — huge standard errors, possibly wrong signs.

The two columns x₁ and x₂ are now nearly parallel. Their combined column space is still a plane, but it's a *paper-thin* plane — nearly degenerate. The projection of y onto this plane still works (R² is fine), but the exact location of the projection along the thin direction is wildly unstable.

In [ ]:
# Perturb a single observation and watch coefficients swing
print(f"{'Perturbation':>15s}  {'β̂₁':>8s}  {'β̂₂':>8s}  {'β̂₃':>8s}  {'R²':>6s}")
print("-" * 55)

y_sd = np.std(y_high)
for delta in [0.0, 0.1, -0.1, 0.05, -0.05, 0.15]:
    y_pert = y_high.copy()
    y_pert[0] += delta * y_sd
    m = sm.OLS(y_pert, X_high).fit()
    label = "Original" if delta == 0 else f"{delta:+.2f} SD"
    print(f"{label:>15s}  {m.params[1]:8.2f}  {m.params[2]:8.2f}  {m.params[3]:8.2f}  {m.rsquared:6.3f}")

print("\nMove one data point by a hair and β̂₁, β̂₂ swing wildly. R² barely moves.")

The shadow is skating on a knife's edge. The projection ONTO the plane is stable (R² doesn't change). The projection's location WITHIN the thin direction of the plane is not.

In [ ]:
# Eigenvalues of X'X and ellipsoid: well-conditioned vs collinear
ell_low = Ellipsoid(cs_low.X.T @ cs_low.X)
ell_high = Ellipsoid(cs_high.X.T @ cs_high.X)

print(f"Well-conditioned eigenvalues: {ell_low.eigenvalues.round(1)}")
print(f"Collinear eigenvalues:        {ell_high.eigenvalues.round(1)}")
print(f"\nCondition number (low corr):  {ell_low.condition_number:.1f}")
print(f"Condition number (high corr): {ell_high.condition_number:.1f}")

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# Top row: eigenvalue bar charts
for ax, ell, title in zip(axes[0], [ell_low, ell_high],
                           ["r = 0.3 (stable)", "r = 0.98 (degenerate)"]):
    colors = [PROJECTION if ev > 1 else RESIDUAL for ev in ell.eigenvalues]
    ax.bar(range(len(ell.eigenvalues)), ell.eigenvalues, color=colors)
    ax.set(title=title, xlabel="Eigenvalue index", ylabel="Eigenvalue")
    ax.set_xticks(range(len(ell.eigenvalues)))

# Bottom row: ellipsoid cross-sections
for ax, ell, title in zip(axes[1], [ell_low, ell_high],
                           [f"κ ≈ {ell_low.condition_number:.0f}",
                            f"κ ≈ {ell_high.condition_number:.0f}"]):
    theta = np.linspace(0, 2*np.pi, 100)
    a = np.sqrt(ell.eigenvalues[0])
    b = np.sqrt(min(ell.eigenvalues))
    ax.plot(a * np.cos(theta), b * np.sin(theta), color=COLUMN_SPACE, lw=2)
    ax.fill(a * np.cos(theta), b * np.sin(theta), color=COLUMN_SPACE, alpha=SURFACE_ALPHA)
    lim = max(a, 15) * 1.2
    ax.set(title=title, aspect="equal", xlim=(-lim, lim), ylim=(-lim, lim))
    ax.axhline(0, color=SECONDARY, lw=0.5)
    ax.axvline(0, color=SECONDARY, lw=0.5)
    ax.grid(True, alpha=0.3)

plt.suptitle("Eigenvalue bar charts (top) and ellipsoid cross-sections (bottom)", fontsize=TITLE_SIZE)
plt.tight_layout()
plt.show()

The eigenvalues measure the "width" of the column space in each direction. A large eigenvalue means the column space is wide in that direction — the projection is stable. A tiny eigenvalue means the column space is paper-thin — the projection is skating.

The condition number — the ratio of largest to smallest eigenvalue — measures how thin the ellipsoid is. A condition number of 8 means the thinnest direction is about 8 times narrower than the widest. A condition number of 900 means the thinnest direction has nearly vanished. That's when projections become unstable.

In [ ]:
# Interactive: drag r(x1, x2) from 0 to 0.99 and watch the column space collapse
if INTERACTIVE:
    fig = viz.plot_collinearity_slider(n=200, seed=42)
else:
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, rho in zip(axes, [0.0, 0.5, 0.9, 0.99]):
        cov_r = [[1.0, rho, 0.3], [rho, 1.0, 0.3], [0.3, 0.3, 1.0]]
        rng_s = np.random.default_rng(42)
        X_r = rng_s.multivariate_normal(mean, cov_r, size=200)
        y_r = X_r @ beta_true + rng_s.normal(0, 2.0, size=200)
        cs_r = ColumnSpace(X_r, add_intercept=True)
        ell_r = Ellipsoid(cs_r.X.T @ cs_r.X)
        m_r = sm.OLS(y_r, sm.add_constant(X_r)).fit()
        colors_r = [PROJECTION if ev > 1 else RESIDUAL for ev in ell_r.eigenvalues]
        ax.bar(range(len(ell_r.eigenvalues)), ell_r.eigenvalues, color=colors_r)
        ax.set_title(f"r = {rho}\nκ = {ell_r.condition_number:.0f}\nSE(β̂₁) = {m_r.bse[1]:.2f}", fontsize=10)
        ax.set_xticks(range(len(ell_r.eigenvalues)))
    plt.suptitle("As correlation climbs, the smallest eigenvalue vanishes", fontsize=TITLE_SIZE)
    plt.tight_layout()
    plt.show()

# Scoreboard for the collinear model
scoreboard_high = GeometricScoreboard(
    proj=proj_high, cs=cs_high,
    active_gauges=["theta", "r_squared", "leverage", "residual_norm", "kappa"],
    mode="widget" if INTERACTIVE else "static"
)
scoreboard_high.display()

The Scoreboard saw this coming. The condition number went red. The geometric tools worked here — the eigenvalue structure told you the column space was degenerate. Multicollinearity is a problem the geometry can diagnose. You know how to read the warning signs.

Now for the harder question: what happens when the geometry *can't* warn you?

---

Part 1 was about a problem the geometry can see: the column space degenerates and the projection becomes unstable. Part 2 is about a problem the geometry cannot see — and it's the more dangerous one.

---

## Part 2: Omitted Variable Bias — The Geometry Can't Warn You

---
### 🛑 PREDICT FIRST

We're going to show you a regression that looks perfectly healthy geometrically — well-conditioned column space, stable projection, all Scoreboard gauges green, reasonable condition number. But the regression's answer will be deeply misleading.

**Before running the next cell, write your prediction below:**

What could possibly be wrong with a regression whose geometry is fine? Take 60 seconds to think about this before scrolling.

---

In [ ]:
# YOUR PREDICTION:
#
#

In [ ]:
# Load Meridian and run the SHORT model: salary ~ experience + education + gender
df = load_meridian()
X_short = df[['experience', 'education', 'gender']]
X_short_dm = sm.add_constant(X_short)
model_short = sm.OLS(df['salary'], X_short_dm).fit()
print(model_short.summary())

gender_coeff_short = model_short.params['gender']
print(f"\nGender coefficient: ${gender_coeff_short:,.0f}")
print(f"p-value: {model_short.pvalues['gender']:.4f}")

In [ ]:
# Geometric health check — every diagnostic is green
cs_short = ColumnSpace(X_short_dm.values, add_intercept=False)
proj_short = cs_short.project(df['salary'].values)

print(f"Condition number: {cs_short.condition_number():.1f}")
print(f"R²: {proj_short.r_squared:.3f}")
print(f"||e||/||y||: {proj_short.relative_residual_norm:.3f}")

hm_short = HatMatrix(proj_short.H)
h_diag = hm_short.diagonal()
print(f"Max leverage: {h_diag.max():.4f}  (threshold: {2 * cs_short.p / cs_short.n:.4f})")

scoreboard_short = GeometricScoreboard(
    proj=proj_short, cs=cs_short,
    active_gauges=["theta", "r_squared", "leverage", "residual_norm", "kappa"],
    mode="widget" if INTERACTIVE else "static"
)
scoreboard_short.display()

Every geometric tool says this regression is healthy. The column space is well-conditioned. The projection is stable. The residuals are perpendicular. The hat matrix is well-behaved. The condition number is fine.

The regression says women at Meridian earn about $7,500 less than men with similar experience and education. The geometry says this estimate is precise and reliable.

Is it right?

In [ ]:
# The reveal: add department to the regression
X_long = df[['experience', 'education', 'gender']].copy()
dept_dummies = pd.get_dummies(df['department'], drop_first=True, dtype=float)
X_long = pd.concat([X_long, dept_dummies], axis=1)
X_long_dm = sm.add_constant(X_long)
model_long = sm.OLS(df['salary'], X_long_dm).fit()
print(model_long.summary())

gender_coeff_long = model_long.params['gender']
print(f"\nGender coefficient (short model): ${gender_coeff_short:,.0f}")
print(f"Gender coefficient (long model):  ${gender_coeff_long:,.0f}")
shrinkage = abs(gender_coeff_short - gender_coeff_long) / abs(gender_coeff_short) * 100
print(f"\nThe gap shrank by {shrinkage:.0f}%.")

The gender gap just shrank dramatically. Same data. Same employees. Same salaries. The only difference: you added department to the model.

In [ ]:
# Both Scoreboards side by side — both green
cs_long = ColumnSpace(X_long_dm.values, add_intercept=False)
proj_long = cs_long.project(df['salary'].values)

print("SHORT MODEL (without department)")
print(f"  Condition number: {cs_short.condition_number():.1f}")
print(f"  R²: {proj_short.r_squared:.3f}")
print(f"  Gender coeff: ${gender_coeff_short:,.0f}")
print()
print("LONG MODEL (with department)")
print(f"  Condition number: {cs_long.condition_number():.1f}")
print(f"  R²: {proj_long.r_squared:.3f}")
print(f"  Gender coeff: ${gender_coeff_long:,.0f}")

scoreboard_long = GeometricScoreboard(
    proj=proj_long, cs=cs_long,
    active_gauges=["theta", "r_squared", "leverage", "residual_norm", "kappa"],
    mode="widget" if INTERACTIVE else "static"
)
scoreboard_long.display()

# Relevant Triangle for gender in both models
j_short = list(X_short_dm.columns).index('gender')
j_long = list(X_long_dm.columns).index('gender')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rt_short = cs_short.relevant_triangle(df['salary'].values, j_short)
ax = axes[0]
ax.scatter(rt_short['xj_resid'], rt_short['y_resid'], color=SECONDARY, s=10, alpha=0.4)
xs = np.linspace(rt_short['xj_resid'].min(), rt_short['xj_resid'].max(), 100)
ax.plot(xs, rt_short['beta_j'] * xs, color=RESIDUAL, lw=2)
ax.set(title=f"Short model: β_gender = ${rt_short['beta_j']:,.0f}",
       xlabel="gender (residualized)", ylabel="salary (residualized)")
ax.axhline(0, color=SECONDARY, lw=0.5)
ax.axvline(0, color=SECONDARY, lw=0.5)

rt_long = cs_long.relevant_triangle(df['salary'].values, j_long)
ax = axes[1]
ax.scatter(rt_long['xj_resid'], rt_long['y_resid'], color=SECONDARY, s=10, alpha=0.4)
xs = np.linspace(rt_long['xj_resid'].min(), rt_long['xj_resid'].max(), 100)
ax.plot(xs, rt_long['beta_j'] * xs, color=PROJECTION, lw=2)
ax.set(title=f"Long model: β_gender = ${rt_long['beta_j']:,.0f}",
       xlabel="gender (residualized)", ylabel="salary (residualized)")
ax.axhline(0, color=SECONDARY, lw=0.5)
ax.axvline(0, color=SECONDARY, lw=0.5)

plt.suptitle("The Relevant Triangle: same variable, different walls", fontsize=TITLE_SIZE)
plt.tight_layout()
plt.show()

Both Scoreboards are green. Both regressions are geometrically healthy. The geometry was flawless in both cases. And they give you answers that differ by thousands of dollars.

### Why the Geometry Can't See This

The geometry tells you how well you're projecting. It cannot tell you whether you're projecting onto the right wall.

In the short model, you aimed the flashlight at a wall that didn't include department. The shadow fell at a right angle. The projection was optimal. But the wall was incomplete — it was tilted in a way that distorted the gender shadow.

In the long model, you aimed the flashlight at a wider wall that includes department. The shadow fell at a right angle. The projection was optimal. The wall was more complete.

The flashlight worked perfectly both times. The problem was the wall. And the flashlight cannot tell you which wall is correct. For that, you need to know the story of the data — why department matters, how it relates to gender, whether it's a confounder or a mediator.

In [ ]:
# OVB formula: β̂_short = β̂_long + δ·γ
dept_dummies = pd.get_dummies(df['department'], drop_first=True, dtype=float)
dept_cols = list(dept_dummies.columns)
X_gamma = sm.add_constant(df[['experience', 'education', 'gender']])

# δ: effect of each department on salary (from long model)
delta = np.array([model_long.params[c] for c in dept_cols])

# γ: how gender predicts each department (controlling for experience, education)
gamma = np.array([
    sm.OLS(dept_dummies[c], X_gamma).fit().params['gender']
    for c in dept_cols
])

bias = delta @ gamma

print(f"β̂_short (gender) = ${gender_coeff_short:,.0f}")
print(f"β̂_long  (gender) = ${gender_coeff_long:,.0f}")
print(f"δ·γ (the bias)   = ${bias:,.0f}")
print(f"β̂_long + δ·γ     = ${gender_coeff_long + bias:,.0f}")
print(f"\nThe OVB formula works: β̂_short ≈ β̂_long + δ·γ")

The bias δ·γ is the gap between the two projections. It exists because department is correlated with both gender (γ ≠ 0) and salary (δ ≠ 0). When you omit department, its effect on salary gets absorbed into the gender coefficient.

The geometry didn't lie — it accurately projected onto the column space you gave it. But the column space you gave it was missing a piece, and that missing piece was correlated with both the predictor you care about and the outcome.

### The "Which Wall?" Exercise

Here are two walls — two column spaces. One includes department, one doesn't.

Which wall should you aim the flashlight at?

The geometry can't tell you. Both projections are valid. Both are perpendicular. Both are optimal given their column space. To choose between them, you need to answer a question that is not geometric: is department a **confounder** (something that causes both gender composition and salary) or a **mediator** (something that gender causes, which then causes salary differences)?

- If department is a **confounder**: control for it. Use the wider wall. The gender gap is small.
- If department is a **mediator** (gender → department → salary): don't control for it. The wider wall removes part of the effect you're trying to measure. The gender gap might be larger.

This is a causal question, not a geometric one. The geometry tells you the *how*. The causal story tells you the *what*.

---
### 🛑 DIAGNOSE FIRST

Here's a regression from a different company. The summary looks similar to Meridian's. The geometry is clean. The Scoreboard is green.

```
                  coef    std err   t      P>|t|
const          42000.0    1200.0  35.00    0.000
experience      1850.0     120.0  15.42    0.000
education       2200.0     180.0  12.22    0.000
remote_work    -3200.0     950.0  -3.37    0.001

R² = 0.62    Condition number: 18.4
```

**Before scrolling, answer:**

Can the geometric framework alone tell you whether this regression has omitted variable bias? What would you need *in addition to* the geometry to answer that question?

---

In [ ]:
# YOUR ANSWER:
#
#

### What You Need Beyond Geometry

You need a causal model. Domain knowledge. A theory about what generates the data. The geometry tells you about the projection. The causal model tells you whether the projection answers your question.

This course has given you powerful geometric tools. This notebook is their honest boundary marker: those tools have a limit, and beyond that limit, you need different tools — directed acyclic graphs, potential outcomes frameworks, natural experiments, instrumental variables. Notebook 10 will name these tools. This notebook's job is to make you feel why you need them.

In [ ]:
# The structural relationships in the Meridian data
print("DEPARTMENT COMPOSITION BY GENDER")
print("=" * 50)
cross = pd.crosstab(df['department'], df['gender'], normalize='index')
cross.columns = ['Male', 'Female']
cross['Total'] = df.groupby('department').size()
print(cross.round(2).to_string())

print("\nMEAN SALARY BY DEPARTMENT")
print("=" * 50)
dept_salary = df.groupby('department')['salary'].mean().sort_values(ascending=False)
for dept, sal in dept_salary.items():
    print(f"  {dept:15s}  ${sal:>10,.0f}")

At Meridian, higher-paying departments tend to have more men. Lower-paying departments tend to have more women. When you omit department, the higher pay in those departments gets partially attributed to being male. The apparent "gender gap" was partly a "department gap" wearing a gender costume.

Was department a confounder or a mediator? If Meridian's hiring practices systematically steer women away from higher-paying departments, then department *is* the mechanism through which gender discrimination operates — and controlling for it removes the very thing you're trying to measure. The answer depends on what question you're asking and what you believe about the causal structure. The geometry has no opinion.

---
### ✍️ The Memo

You're writing a memo to Meridian's CEO. The previous analyst reported a $7,500 gender pay gap. Your analysis, controlling for department, shows approximately $1,100. The CEO asks: "So there's no gender pay gap?"

In 3–5 sentences, write your response.

**Rules:** Do not use the words "omitted variable bias," "confounding," "projection," "column space," or "Simpson's paradox." Write in plain English a non-technical manager would understand.

---

In [ ]:
# YOUR MEMO:
#
#
#

In [ ]:
# Final Scoreboard comparison — both models, both green
print(f"SCOREBOARD: Short Model (gender coeff ≈ ${gender_coeff_short:,.0f})")
scoreboard_short.display()
print()
print(f"SCOREBOARD: Long Model (gender coeff ≈ ${gender_coeff_long:,.0f})")
scoreboard_long.display()

The Scoreboard could not have warned you. Both regressions show healthy geometry. The instruments read green for a regression that estimates a large gender gap and green for a regression that estimates a small one. The best diagnostic panel in the world cannot replace subject-matter knowledge about what belongs in your model.

---
### 🔗 Back to Statsmodels

| Geometric concept | Code |
|---|---|
| Eigenvalues of X'X | `np.linalg.eigvalsh(X.T @ X)` |
| Condition number | `np.linalg.cond(X)` or `cs.condition_number()` |
| Condition number from summary | `model.summary()` (bottom right of table) |
| Compare nested models | Fit both, compare `model.params` |

---

In [ ]:
# Connecting eigenvalues to statsmodels and numpy
X_mat = X_short_dm.values
eigvals_numpy = np.linalg.eigvalsh(X_mat.T @ X_mat)

print(f"Eigenvalues (numpy):        {np.sort(eigvals_numpy)[::-1].round(1)}")
print(f"Eigenvalues (ColumnSpace):  {cs_short.eigenvalues().round(1)}")
print(f"\nCondition number (ColumnSpace): {cs_short.condition_number():.1f}")
print(f"Condition number (numpy, X'X):  {eigvals_numpy.max() / eigvals_numpy.min():.1f}")
print(f"\nStatsmodels reports the condition number at the bottom of model.summary().")
print(f"When it exceeds 30, check for multicollinearity.")
print(f"When it exceeds 100, individual coefficients may be unreliable.")
print(f"But remember: a healthy condition number cannot detect omitted variables.")

---
### Summary

**What you learned:**
- Multicollinearity makes the column space degenerate — the geometry sees this and warns you through the condition number and eigenvalue structure.
- Omitted variable bias makes the column space incomplete — the geometry cannot see this because both projections are valid.
- The OVB formula β̂_short = β̂_long + δ·γ measures the gap between two projections onto different walls.

**Key geometric insight:** ***The geometry tells you how well you're projecting. It cannot tell you whether you're projecting onto the right wall.***

### Next

If the column space is unstable (multicollinearity), can you deliberately constrain the projection to stabilize it? Notebook 7 introduces regularization — Ridge and LASSO — and reveals their own beautiful geometry.

---